# 04 · Entrenamiento supervisado: clasificación

La prueba compara una regresión logística como baseline frente a Random Forest, Extra Trees y XGBoost. Usa GroupKFold por conglomerado para elegir modelo y una partición agrupada separada para la evaluación final. El escalado, imputación y codificación se ajustan dentro de cada fold.

In [ ]:
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pipeline.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("No se encontró pipeline.py.")
if not (ROOT / "data" / "processed" / "dataset_enaho_escale_2025.parquet").is_file():
    raise FileNotFoundError("Falta la base integrada; ejecuta la integración o pipeline.py.")

## Entrenar y probar modelos

La etiqueta disponible es no uso individual reportado de Internet entre quienes asisten. Se combina FACTOR07 con pesos de clase en el entrenamiento. Se comparan ROC-AUC, PR-AUC y F1 macro, ponderados por el factor muestral.

In [ ]:
subprocess.run([sys.executable, str(ROOT / "src" / "07_modelar_clasificacion.py")], cwd=ROOT, check=True)

In [ ]:
OUT = ROOT / "informes" / "resultados_clasificacion"
cv = pd.read_csv(OUT / "comparacion_modelos_groupkfold.csv")
test = pd.read_csv(OUT / "comparacion_modelos_prueba.csv")
display(cv[["modelo", "roc_auc_ponderado_mean", "pr_auc_ponderado_mean", "f1_macro_ponderado_mean"]])
display(test[["modelo", "seleccionado_por_cv", "roc_auc_ponderado", "pr_auc_ponderado", "f1_macro_ponderado", "interseccion_conglomerados"]])
selected = test.loc[test["seleccionado_por_cv"], "modelo"].tolist()
if len(selected) != 1:
    raise ValueError("Debe haber exactamente un modelo seleccionado por CV.")
print("Ganador de validación:", selected[0])